# Session 13: 20M language model and reversible training

This notebook runs four experiments on one Colab GPU: a standard baseline, midpoint and Hamiltonian symplectic-Euler reversibility at one shared batch size, then the better reversible variant at its largest safe batch. Every run processes exactly 50M training tokens.

Set **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU) before running. The complete experiment trains 200M tokens total and can take a while. Results are written to Google Drive when mounted, including tokenized-data cache, JSON/CSV metrics, a Markdown report, and a loss plot.

In [ ]:
%pip -q install datasets tiktoken matplotlib
import sys
from pathlib import Path

repo = Path('/content/ERA5')
if not (repo / 'session13' / 'reversible_lm.py').exists():
    import subprocess
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/gmrammohan15/ERA5.git', str(repo)], check=True)
sys.path.insert(0, str(repo))

import torch
from session13.reversible_lm import (
    Config, load_fineweb_tokens, load_token_streams,
    run_experiment, run_smoke_checks, save_token_streams
)

assert torch.cuda.is_available(), 'Select a CUDA GPU runtime before training.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('Default parameter count:', Config().parameter_count())

In [ ]:
# Small correctness checks run on CPU and take only a few seconds.
smoke = run_smoke_checks()
smoke

In [ ]:
# Persist the tokenized stream so later runs can skip re-tokenizing it.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    output_dir = Path('/content/drive/MyDrive/session13_artifacts')
except ImportError:
    output_dir = Path('/content/session13_artifacts')
output_dir.mkdir(parents=True, exist_ok=True)
token_cache = output_dir / 'fineweb_edu_gpt2_tokens.pt'
config = Config()
if token_cache.exists():
    streams = load_token_streams(token_cache)
    print('Loaded token cache:', token_cache)
else:
    streams = load_fineweb_tokens(config)
    save_token_streams(streams, token_cache)
    print('Saved token cache:', token_cache)
assert len(streams.train) == config.train_tokens, 'Cached training token count does not match this config.'
assert len(streams.validation) >= config.validation_tokens, 'Cached validation stream is too short.'
print(f'Train tokens: {len(streams.train):,}; validation tokens: {len(streams.validation):,}')

## Train and compare

The fixed comparison batch is calibrated against the standard baseline with 12% GPU-memory headroom. The reversible runs reuse that batch size. The notebook selects the reversible architecture with the lower final validation loss, then calibrates the largest safe batch for that variant. Calibration is a single forward/backward/update probe; reported metrics come from full 50M-token runs.

In [ ]:
experiment = run_experiment(
    config=config,
    streams=streams,
    device=torch.device('cuda:0'),
    output_dir=output_dir,
    maximum_batch_size=1024,
)
print('Selected reversible variant:', experiment['selected_reversible_variant'])
print('Results:', output_dir / 'REPORT.md')

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(output_dir / 'loss_curves.png')))
print((output_dir / 'REPORT.md').read_text())